# English → Hindi Transformer (Colab GPU)

1. **Runtime → Change runtime type → T4 GPU** (or better)
2. Upload this folder's `.py` files and `Dataset_English_Hindi.csv` to Drive (e.g. `MyDrive/en_hi_transformer/`), or just upload them to Colab's file panel.
3. Run the cells top to bottom.

In [ ]:
!nvidia-smi
!pip install -q datasets tokenizers tensorboard tqdm

## Mount Drive (recommended — checkpoints survive disconnects)
Set `PROJECT` to the Drive folder holding the `.py` files and the CSV.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/en_hi_transformer'   # change if needed
%cd $PROJECT
!ls

In [ ]:
from config import get_config
from train import train_model

cfg = get_config()
cfg['data_file']       = f'{PROJECT}/Dataset_English_Hindi.csv'
cfg['model_folder']    = f'{PROJECT}/weights'
cfg['tokenizer_file']  = f'{PROJECT}/tokenizer_{{0}}.json'
cfg['experiment_name'] = f'{PROJECT}/runs/tmodel'

# GPU tuning: T4 -> batch 64 | L4/A100 -> try 128 or 256
# cfg['batch_size'] = 128
# cfg['num_epoch']  = 20

train_model(cfg)   # re-run this cell after a disconnect: it resumes from the latest checkpoint

## (Optional) TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $PROJECT/runs

## Translate with the trained model

In [ ]:
import torch
from translate import load, translate

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, tok_src, tok_tgt = load(cfg, dev)
for s in ['How are you?', 'I am going to school.']:
    print(s, '->', translate(s, model, tok_src, tok_tgt, cfg, dev))